In [2]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import config
from audit_checks import check_csv, polygon_ids, street_rows

DUMP = ROOT / "data" / "raw" / "fahrraddiebstahl_2026-10-26.csv"
df = check_csv(DUMP)
df["LOR"] = df["LOR"].str.zfill(8)
street = street_rows(df).copy()
areas = sorted(polygon_ids())
print(f"Rows: {len(df):,} | street: {len(street):,} | cellar: {len(df) - len(street):,}")
print(f"Areas with any record: {df['LOR'].nunique()} of {len(areas)} | with a street record: {street['LOR'].nunique()}")

Encoding: iso-8859-1, delimiter: ',', rows: 31,397
Rows: 31,397 | street: 29,796 | cellar: 1,601
Areas with any record: 535 of 542 | with a street record: 534


In [3]:
for col in ["ANGELEGT_AM", "TATZEIT_ANFANG_DATUM", "TATZEIT_ENDE_DATUM"]:
    street[col] = pd.to_datetime(street[col], format="%d.%m.%Y")
lag = (street["ANGELEGT_AM"] - street["TATZEIT_ENDE_DATUM"]).dt.days
lag = lag[lag >= 0]
print("Reporting lag p50 / p90 / p95 (days):", *lag.quantile([0.5, 0.9, 0.95]).round().astype(int))
data_to = street["ANGELEGT_AM"].max()
print(f"Offences from {street['TATZEIT_ANFANG_DATUM'].min():%d %b %Y}, reports up to {data_to:%d %b %Y}")

Reporting lag p50 / p90 / p95 (days): 0 4 10
Offences from 01 Jan 2025, reports up to 24 Oct 2026


In [4]:
year = street[street["TATZEIT_ANFANG_DATUM"] > data_to - pd.Timedelta(days=365)].copy()
start_hour = pd.to_numeric(year["TATZEIT_ANFANG_STUNDE"], errors="coerce")
end_hour = pd.to_numeric(year["TATZEIT_ENDE_STUNDE"], errors="coerce")


def daypart(h):
    if pd.isna(h):
        return None
    if 6 <= h < 11:
        return "Morning"
    if 11 <= h < 17:
        return "Afternoon"
    if 17 <= h < 22:
        return "Evening"
    return "Night"


year["daypart"] = start_hour.apply(daypart)
year["daytype"] = year["TATZEIT_ANFANG_DATUM"].dt.weekday.apply(config.daytype_of)
year["week_part"] = (year["TATZEIT_ANFANG_DATUM"].dt.weekday >= 5).map({True: "Sat-Sun", False: "Mon-Fri"})
value = pd.to_numeric(year["SCHADENSHOEHE"], errors="coerce")
year["value_band"] = (value >= 1000).map({True: ">= EUR 1,000", False: "< EUR 1,000"})

start = year["TATZEIT_ANFANG_DATUM"] + pd.to_timedelta(start_hour, unit="h")
end = year["TATZEIT_ENDE_DATUM"] + pd.to_timedelta(end_hour, unit="h")
window = (end - start).dt.total_seconds() / 3600
print(f"12-month street thefts: {len(year):,}")
print("Offence window <= 1 h / 4 h / 24 h:", *[f"{(window <= h).mean():.0%}" for h in (1, 4, 24)])
for col in ["daypart", "daytype", "week_part", "value_band"]:
    print(year[col].value_counts(normalize=True).map("{:.0%}".format).to_string(), "\n")

12-month street thefts: 15,684
Offence window <= 1 h / 4 h / 24 h: 20% 38% 86%
daypart
Evening      38%
Afternoon    30%
Morning      21%
Night        11% 

daytype
Mon_Thu    59%
Weekend    25%
Friday     16% 

week_part
Mon-Fri    75%
Sat-Sun    25% 

value_band
< EUR 1,000     54%
>= EUR 1,000    46% 



In [5]:
counts = year.groupby("LOR").size().reindex(areas, fill_value=0)
print(f"Per area: median {counts.median():.0f}; areas < 10: {(counts < 10).sum()}; < 5: {(counts < 5).sum()}; 0: {(counts == 0).sum()}")


def density(cols):
    levels = [areas] + [sorted(year[c].dropna().unique()) for c in cols]
    full = pd.MultiIndex.from_product(levels, names=["LOR"] + cols)
    n = year.groupby(["LOR"] + cols).size().reindex(full, fill_value=0)
    return {"cells": len(n), "median per cell": n.median(), "cells < 5": f"{(n < 5).mean():.0%}", "cells = 0": f"{(n == 0).mean():.0%}"}


splits = {"daypart": ["daypart"], "daytype (3)": ["daytype"], "Mon-Fri / Sat-Sun": ["week_part"],
          "value band": ["value_band"], "daypart x daytype": ["daypart", "daytype"]}
print(pd.DataFrame({name: density(cols) for name, cols in splits.items()}).T.to_string())

Per area: median 18; areas < 10: 176; < 5: 88; 0: 13
                  cells median per cell cells < 5 cells = 0
daypart            2168             4.0       55%       15%
daytype (3)        1626             5.0       47%       13%
Mon-Fri / Sat-Sun  1084             7.0       36%        8%
value band         1084             9.0       33%        7%
daypart x daytype  6504             1.0       84%       39%


In [6]:
print(year["ART_DES_FAHRRADS"].value_counts().to_string(), "\n")
big = counts[counts >= 20].index
g = year[year["LOR"].isin(big)].groupby("LOR")
defeated = g["ERFASSUNGSGRUND"].apply(lambda s: s.str.contains("schwerer").mean())
median_value = g["SCHADENSHOEHE"].apply(lambda s: pd.to_numeric(s, errors="coerce").median())
print(f"Areas with >= 20 thefts: {len(big)}")
print(f"Lock defeated: citywide {year['ERFASSUNGSGRUND'].str.contains('schwerer').mean():.0%}; per area p10-p90 {defeated.quantile(0.1):.0%} to {defeated.quantile(0.9):.0%}")
print(f"Median stolen value: citywide EUR {value.median():,.0f}; per area p10-p90 EUR {median_value.quantile(0.1):,.0f} to {median_value.quantile(0.9):,.0f}")
print(f"Areas with an expected 30-day count >= 3: {(counts * 30 / 365 >= 3).sum()}")

ART_DES_FAHRRADS
Herrenfahrrad        4429
Fahrrad              2587
Damenfahrrad         2267
Mountainbike         1384
Citybike             1291
Trekkingrad          1258
Rennrad              1072
Kinderfahrrad         711
diverse Fahrräder     254
Hollandrad            149
Lastenfahrrad         141
Klapprad              130
BMX                    10
Liegerad                1 

Areas with >= 20 thefts: 255
Lock defeated: citywide 94%; per area p10-p90 90% to 100%
Median stolen value: citywide EUR 900; per area p10-p90 EUR 653 to 1,298
Areas with an expected 30-day count >= 3: 145
